# 作业 3：面向表格和时间序列基础模型的多位鲁棒水印方案

## 研究动机

现有 LLM 水印方法 (KGW, Unbiased 等) 主要针对自回归文本生成设计，依赖 token 级别的统计偏置。
但对于**表格基础模型** (tabular foundation models) 和**时间序列基础模型** (time-series foundation models)，
其输出是结构化数值序列，不经过 token 采样过程，因此传统方法不适用。

## 方案设计

本作业提出一种 **结构化残差水印 (Structured Residual Watermarking)** 方案，
将 KGW 的"密钥控制统计偏置"思想迁移到结构化数据领域：

1. **编码**：使用密钥哈希函数将每个数据行映射到特定 payload bit，在残差方向上嵌入极小幅度的正/负偏置
2. **多位编码**：支持嵌入 12-bit payload，可实现来源追踪（4096 个可能的身份标识）
3. **检测**：对每个 bit 独立进行分组 Z-test，恢复完整 payload
4. **隐蔽性**：扰动强度控制在原始数据标准差的 3%，保持统计分布不变

## 文献启发

- KGW (Kirchenbauer et al., 2023)：密钥控制的统计偏置 + Z-test 检测
- Distortion-free Watermark (Thickstun, 2023)：最小化对原始分布的扭曲
- MarkLLM 评估框架 (Pan et al., 2024)：多维度评估（检测、质量、鲁棒性、开销）

## 参考文献

- Kirchenbauer, J. et al. "A Watermark for Large Language Models." ICML 2023.
- Thickstun, J. "Robust Distortion-free Watermarks for Language Models." TMLR 2023.
- Pan, L. et al. "MarkLLM: An Open-Source Toolkit for LLM Watermarking." EMNLP 2024.
- 相关时间序列/表格水印文献可参照 Time-series Watermarking 和 Tabular Data Watermarking 的最新研究。


## 1. 实验环境

本作业使用合成时间序列模拟表格或时间序列基础模型的输出，不依赖额外模型下载。
生成 48 个 entity × 150 时间步的合成数据集，总计 7200 行数据。

In [1]:
import json, math, time, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10,
    'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'
})

RUN_DIR = Path.cwd()
SUBMISSION_DIR = RUN_DIR.parent if RUN_DIR.name == 'notebooks' else RUN_DIR / 'submission'
OUT_DIR = SUBMISSION_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(2026)

N_ENTITIES = 48
SERIES_LENGTH = 150
PAYLOAD_BITS = np.array([1,0,1,1,0,1,0,0,1,0,1,0], dtype=int)  # 12-bit payload

print(f'Entities: {N_ENTITIES}, Series length: {SERIES_LENGTH}')
print(f'Total rows: {N_ENTITIES * SERIES_LENGTH}')
print(f'Payload: {PAYLOAD_BITS.tolist()} ({len(PAYLOAD_BITS)} bits)')

def stable_hash_int(value):
    return int(hashlib.sha256(str(value).encode('utf-8')).hexdigest(), 16) % (2**32)


Entities: 48, Series length: 150
Total rows: 7200
Payload: [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0] (12 bits)


## 2. 合成时间序列数据生成

每个 entity 包含：
- $\mathbf{T}$：线性趋势分量 (不同 entity 有不同的趋势斜率)
- $\mathbf{S}_1$：低频季节项 (周期 $\sim 30$)
- $\mathbf{S}_2$：高频周期项 (周期 $\sim 7.5$, 模拟周效应)
- $\boldsymbol{\epsilon}$：高斯噪声 ($\sigma=0.18$)

合成数据公式：
$$y_{i,t} = 10 + 0.25i + T_t + \sin\left(\frac{5\pi t}{L} + 0.12i\right) + 0.25\sin\left(\frac{20\pi t}{L} + i\right) + \epsilon_{i,t}$$

其中 $i$ 为 entity 索引，$t$ 为时间步，$L$ 为序列长度。

In [2]:
def make_synthetic_series(n_entities=N_ENTITIES, length=SERIES_LENGTH):
    rows = []
    for entity in range(n_entities):
        t = np.arange(length)
        trend = np.linspace(0, 1.0 + 0.02 * entity, length)
        season = np.sin(np.linspace(0, 5*np.pi, length) + entity * 0.12)
        weekly = 0.25 * np.sin(np.linspace(0, 20*np.pi, length) + entity)
        noise = RNG.normal(0, 0.18, length)
        value = 10 + entity * 0.25 + trend + season + weekly + noise
        group = ['A', 'B', 'C'][entity % 3]
        segment = ['train', 'val', 'test'][entity % 3]
        for t_idx, y in enumerate(value):
            rows.append({'entity': entity, 'time': t_idx, 'group': group, 
                         'segment': segment, 'value': float(y)})
    return pd.DataFrame(rows)

base_df = make_synthetic_series()
base_df.to_csv(OUT_DIR / 'assignment3_base_series.csv', index=False, encoding='utf-8-sig')
print(f"Base data: {len(base_df)} rows, {base_df['entity'].nunique()} entities")
print(f"Value stats: mean={base_df['value'].mean():.3f}, std={base_df['value'].std():.3f}, "
      f"min={base_df['value'].min():.3f}, max={base_df['value'].max():.3f}")
base_df.head()


Base data: 7200 rows, 48 entities
Value stats: mean=16.599, std=3.684, min=8.761, max=25.094


,entity,time,group,segment,value
0,0,0,A,train,9.857238
1,0,1,A,train,10.257567
2,0,2,A,train,10.068094
3,0,3,A,train,10.820802
4,0,4,A,train,10.799363


## 3. 多位水印嵌入

### 3.1 嵌入算法

对于数据表中的每一行 $(i, t)$：
1. 使用密钥哈希函数 $H_K(i, t)$ 确定该行对应的 payload bit 索引 $b \in \{0, ..., B-1\}$
2. 使用独立哈希确定扰动方向 $s \in \{-1, +1\}$
3. 根据目标 bit 值确定目标方向：$d = s$ if $\text{payload}[b]=1$ else $-s$
4. 嵌入扰动：$\tilde{y} = y + d \cdot \delta \cdot \sigma_y + \eta$，其中 $\eta \sim \mathcal{N}(0, \sigma_j \cdot \sigma_y)$

其中 $\delta=0.03$ 为水印强度，$\sigma_j=0.014$ 为抖动幅度，$\sigma_y$ 为原始数据标准差。

### 3.2 检测算法

1. 计算残差 $r = \tilde{y} - y$
2. 对每个 bit，收集所有分配给该 bit 的行的残差，乘以对应的方向符号
3. 进行单样本 Z-test：$Z_b = \frac{\bar{r}_b}{\text{SE}(r_b)}$
4. 若 $Z_b > 0$ 则恢复 bit 为 1，否则为 0

In [3]:
def embed_structured_watermark(df, bits, key='course-final-2026', strength=0.03, jitter=0.014):
    """Embed multi-bit watermark into structured residual."""
    out = df.copy()
    sigma = out['value'].std()
    out['watermarked_value'] = out['value'].astype(float)
    out['bit_id'] = -1
    out['target_direction'] = 0
    out['perturbation'] = 0.0
    
    for idx in out.index:
        row = out.loc[idx]
        bit_id = stable_hash_int((key, int(row['entity']), int(row['time']))) % len(bits)
        sign = 1 if stable_hash_int((key, bit_id, 'sign')) % 2 == 0 else -1
        target = sign if bits[bit_id] == 1 else -sign
        perturbation = target * strength * sigma + RNG.normal(0, jitter * sigma)
        out.at[idx, 'watermarked_value'] = float(row['value']) + perturbation
        out.at[idx, 'bit_id'] = bit_id
        out.at[idx, 'target_direction'] = target
        out.at[idx, 'perturbation'] = perturbation
    
    return out

def detect_structured_watermark(df, bits_len, key='course-final-2026', 
                                 value_col='watermarked_value', reference_col='value'):
    """Detect multi-bit watermark via per-bit Z-test.
    Resets index to handle non-sequential indices from sampled/subset data."""
    work = df.reset_index(drop=True)
    residual = work[value_col].to_numpy() - work[reference_col].to_numpy()
    rows = []
    for bit_id in range(bits_len):
        idxs, signs = [], []
        for i, row in work.iterrows():
            b = stable_hash_int((key, int(row['entity']), int(row['time']))) % bits_len
            if b == bit_id:
                sign = 1 if stable_hash_int((key, bit_id, 'sign')) % 2 == 0 else -1
                idxs.append(i)
                signs.append(sign)
        vals = residual[idxs] * np.array(signs)
        if len(vals) > 1:
            z = vals.mean() / (vals.std(ddof=1) / np.sqrt(len(vals)) + 1e-9)
        else:
            z = 0.0
        rows.append({'bit_id': bit_id, 'z_score': z, 
                     'recovered_bit': int(z > 0), 'samples': len(vals)})
    return pd.DataFrame(rows)

# Embed watermark
wm_df = embed_structured_watermark(base_df, PAYLOAD_BITS)
detect_df = detect_structured_watermark(wm_df, len(PAYLOAD_BITS))
bit_accuracy = accuracy_score(PAYLOAD_BITS, detect_df['recovered_bit'])
wm_df.to_csv(OUT_DIR / 'assignment3_full_watermarked_series.csv', index=False, encoding='utf-8-sig')
detect_df.to_csv(OUT_DIR / 'assignment3_full_bit_detection.csv', index=False, encoding='utf-8-sig')

print(f'Payload bits:    {PAYLOAD_BITS.tolist()}')
print(f'Recovered bits:  {detect_df["recovered_bit"].tolist()}')
print(f'Bit accuracy:    {bit_accuracy:.4f} ({bit_accuracy*len(PAYLOAD_BITS):.0f}/{len(PAYLOAD_BITS)} correct)')
print(f'Mean |Z|:        {detect_df["z_score"].abs().mean():.2f}')
print(f'Min |Z|:         {detect_df["z_score"].abs().min():.2f}')
print(f'Detection margin: excellent (all |Z| >> 3)')
display(detect_df.round(3))

Payload bits:    [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0]
Recovered bits:  [1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0]
Bit accuracy:    1.0000 (12/12 correct)
Mean |Z|:        52.45
Min |Z|:         48.88
Detection margin: excellent (all |Z| >> 3)


,bit_id,z_score,recovered_bit,samples
0,0,52.448,1,641
1,1,-52.125,0,595
2,2,53.000,1,611
3,3,48.875,1,577
4,4,-50.371,0,578
5,5,52.176,1,589
6,6,-50.681,0,571
7,7,-53.240,0,612
8,8,53.428,1,610
9,9,-54.039,0,565


## 4. 质量损失分析

从多个维度量化水印对数据质量的影响：
- 基本统计量（均值、标准差、极值、分位数）
- Pearson 相关系数
- Kolmogorov-Smirnov 检验（分布一致性）
- 扰动幅度分布

In [4]:
# Comprehensive quality metrics
quality_rows = []
for metric, original, watermarked in [
    ('mean', base_df['value'].mean(), wm_df['watermarked_value'].mean()),
    ('std', base_df['value'].std(), wm_df['watermarked_value'].std()),
    ('min', base_df['value'].min(), wm_df['watermarked_value'].min()),
    ('max', base_df['value'].max(), wm_df['watermarked_value'].max()),
    ('q05', base_df['value'].quantile(0.05), wm_df['watermarked_value'].quantile(0.05)),
    ('q25', base_df['value'].quantile(0.25), wm_df['watermarked_value'].quantile(0.25)),
    ('q50', base_df['value'].quantile(0.50), wm_df['watermarked_value'].quantile(0.50)),
    ('q75', base_df['value'].quantile(0.75), wm_df['watermarked_value'].quantile(0.75)),
    ('q95', base_df['value'].quantile(0.95), wm_df['watermarked_value'].quantile(0.95)),
    ('skewness', base_df['value'].skew(), wm_df['watermarked_value'].skew()),
    ('kurtosis', base_df['value'].kurtosis(), wm_df['watermarked_value'].kurtosis()),
]:
    rel_change = abs(watermarked - original) / (abs(original) + 1e-9)
    quality_rows.append({
        'metric': metric, 'original': original, 'watermarked': watermarked,
        'absolute_change': abs(watermarked - original), 'relative_change': rel_change
    })

# Correlation
pearson_r = base_df['value'].corr(wm_df['watermarked_value'])
spearman_r = base_df['value'].corr(wm_df['watermarked_value'], method='spearman')
quality_rows.append({'metric': 'pearson_corr', 'original': 1.0, 'watermarked': pearson_r, 
                     'absolute_change': 1.0 - pearson_r, 'relative_change': (1.0 - pearson_r)})
quality_rows.append({'metric': 'spearman_corr', 'original': 1.0, 'watermarked': spearman_r,
                     'absolute_change': 1.0 - spearman_r, 'relative_change': (1.0 - spearman_r)})

# KS test
ks_stat, ks_p = stats.ks_2samp(base_df['value'], wm_df['watermarked_value'])
quality_rows.append({'metric': 'ks_pvalue', 'original': 1.0, 'watermarked': ks_p,
                     'absolute_change': ks_stat, 'relative_change': 0.0})

quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(OUT_DIR / 'assignment3_full_quality_metrics.csv', index=False, encoding='utf-8-sig')
print("Quality Metrics:")
display(quality_df.round(6))
print(f"\nPerturbation std: {wm_df['perturbation'].std():.6f}")
print(f"Perturbation mean: {wm_df['perturbation'].mean():.10f}")
print(f"Perturbation / data_std: {wm_df['perturbation'].std() / base_df['value'].std():.4f}")


Quality Metrics:


,metric,original,watermarked,absolute_change,relative_change
0,mean,16.599227,16.654519,0.055292,0.003331
1,std,3.684152,3.686710,0.002557,0.000694
2,min,8.760678,8.908885,0.148207,0.016917
3,max,25.094099,25.129480,0.035381,0.001410
4,q05,10.972703,11.025897,0.053194,0.004848
5,q25,13.498875,13.551215,0.052340,0.003877
6,q50,16.488898,16.546837,0.057938,0.003514
7,q75,19.676697,19.726667,0.049969,0.002540
8,q95,22.450874,22.522677,0.071803,0.003198
9,skewness,0.072991,0.072276,0.000715,0.009793



Perturbation std: 0.107939
Perturbation mean: 0.0552916001
Perturbation / data_std: 0.0293


## 5. 鲁棒性攻击评估

评估三类攻击对 payload 恢复的影响：

1. **加性噪声攻击**：在带水印数据上叠加不同强度的高斯噪声 $\mathcal{N}(0, \alpha \cdot \sigma_y)$
2. **随机删行攻击**：随机删除不同比例的数据行，模拟数据缺失
3. **平滑攻击**：对每个 entity 的时间序列应用不同窗口大小的移动平均，模拟数据后处理
4. **量化攻击**：将数据舍入到不同精度，模拟数据压缩/存储格式转换

In [5]:
attack_rows = []
sigma_y = wm_df['value'].std()

# 1. Noise attack
for noise_scale in [0.0, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.18]:
    attacked = wm_df.copy()
    attacked['attacked_value'] = attacked['watermarked_value'] + RNG.normal(0, noise_scale * sigma_y, len(wm_df))
    det = detect_structured_watermark(attacked, len(PAYLOAD_BITS), value_col='attacked_value')
    attack_rows.append({
        'attack': 'additive_noise', 'strength': noise_scale,
        'bit_accuracy': accuracy_score(PAYLOAD_BITS, det['recovered_bit']),
        'mean_abs_z': det['z_score'].abs().mean(), 'min_abs_z': det['z_score'].abs().min()
    })

# 2. Row drop attack
for drop_ratio in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    attacked = wm_df.sample(frac=1-drop_ratio, random_state=int(1000*drop_ratio)+1).sort_values(['entity','time']).copy()
    det = detect_structured_watermark(attacked, len(PAYLOAD_BITS))
    attack_rows.append({
        'attack': 'row_drop', 'strength': drop_ratio,
        'bit_accuracy': accuracy_score(PAYLOAD_BITS, det['recovered_bit']),
        'mean_abs_z': det['z_score'].abs().mean(), 'min_abs_z': det['z_score'].abs().min()
    })

# 3. Smoothing attack
for window in [1, 3, 5, 7, 9, 11, 15]:
    attacked = wm_df.copy()
    attacked['attacked_value'] = attacked.groupby('entity')['watermarked_value'].transform(
        lambda s: s.rolling(window, min_periods=1, center=True).mean())
    det = detect_structured_watermark(attacked, len(PAYLOAD_BITS), value_col='attacked_value')
    attack_rows.append({
        'attack': 'smoothing', 'strength': window,
        'bit_accuracy': accuracy_score(PAYLOAD_BITS, det['recovered_bit']),
        'mean_abs_z': det['z_score'].abs().mean(), 'min_abs_z': det['z_score'].abs().min()
    })

# 4. Quantization attack
for decimals in [3, 2, 1, 0]:
    attacked = wm_df.copy()
    attacked['attacked_value'] = attacked['watermarked_value'].round(decimals)
    det = detect_structured_watermark(attacked, len(PAYLOAD_BITS), value_col='attacked_value')
    attack_rows.append({
        'attack': 'quantization', 'strength': decimals,
        'bit_accuracy': accuracy_score(PAYLOAD_BITS, det['recovered_bit']),
        'mean_abs_z': det['z_score'].abs().mean(), 'min_abs_z': det['z_score'].abs().min()
    })

attack_df = pd.DataFrame(attack_rows)
attack_df.to_csv(OUT_DIR / 'assignment3_full_attack_results.csv', index=False, encoding='utf-8-sig')
print("Attack Results Summary:")
display(attack_df.round(4))


Attack Results Summary:


,attack,strength,bit_accuracy,mean_abs_z,min_abs_z
0,additive_noise,0.00,1.0000,52.4455,48.8751
1,additive_noise,0.01,1.0000,41.8627,38.4398
2,additive_noise,0.02,1.0000,30.2276,28.3397
3,additive_noise,0.03,1.0000,22.3622,20.7129
4,additive_noise,0.05,1.0000,14.0480,12.8062
5,additive_noise,0.08,1.0000,9.1661,8.0834
6,additive_noise,0.12,1.0000,6.0951,5.0397
7,additive_noise,0.18,1.0000,4.3912,2.5857
8,row_drop,0.00,1.0000,52.4455,48.8751
9,row_drop,0.10,1.0000,49.7694,45.8385


## 6. 综合可视化

六面板图展示：(a) 时间序列对比（原始 vs 水印），(b) Bit 级 Z-score 检测，
(c) 分布保持，(d) 关键统计量变化，(e) 攻击下 payload 恢复率，(f) 攻击下 Z-score 衰减。

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(19, 12))

# (a) Time series: original vs watermarked
ax = axes[0, 0]
for ent in [0, 2]:
    sub = wm_df[wm_df['entity'] == ent]
    ax.plot(sub['time'], sub['value'], alpha=0.6, linewidth=1, label=f'Entity {ent} original')
    ax.plot(sub['time'], sub['watermarked_value'], alpha=0.7, linewidth=0.8, 
            linestyle='--', label=f'Entity {ent} watermarked')
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('(a) Original vs Watermarked Time Series', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

# (b) Bit-level Z-score
ax = axes[0, 1]
bit_colors = ['#4C78A8' if b else '#F58518' for b in PAYLOAD_BITS]
ax.bar(detect_df['bit_id'], detect_df['z_score'], color=bit_colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=1)
ax.axhline(3, color='green', linestyle=':', alpha=0.5, label='|Z|=3')
ax.axhline(-3, color='green', linestyle=':', alpha=0.5)
ax.set_xlabel('Bit ID')
ax.set_ylabel('Z-score')
ax.set_title(f'(b) Bit-level Detection (12-bit payload, 100% accuracy)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis='y')

# (c) Distribution preservation
ax = axes[0, 2]
ax.hist(base_df['value'], bins=50, alpha=0.5, label='Original', color='#4C78A8', density=True)
ax.hist(wm_df['watermarked_value'], bins=50, alpha=0.5, label='Watermarked', color='#F58518', density=True)
ax.set_xlabel('Value')
ax.set_ylabel('Density')
ax.set_title('(c) Distribution Preservation', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (d) Quality metric changes
ax = axes[1, 0]
qplot = quality_df[quality_df['metric'].isin(['mean','std','q25','q50','q75','skewness','kurtosis'])]
ax.barh(qplot['metric'], qplot['relative_change'] * 100, color='#54A24B', edgecolor='white')
ax.set_xlabel('Relative Change (%)')
ax.set_title('(d) Relative Change in Key Statistics', fontweight='bold')
ax.grid(alpha=0.3, axis='x')

# (e) Payload recovery under attacks
ax = axes[1, 1]
styles = {'additive_noise': 'o-', 'row_drop': 's--', 'smoothing': '^:', 'quantization': 'd-.'}
for attack_name, sub in attack_df.groupby('attack'):
    style = styles.get(attack_name, 'o-')
    ax.plot(sub['strength'], sub['bit_accuracy'], style, label=attack_name, linewidth=2, markersize=7)
ax.set_ylim(-0.05, 1.08)
ax.set_xlabel('Attack Strength')
ax.set_ylabel('Payload Bit Accuracy')
ax.set_title('(e) Payload Recovery under Attacks', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# (f) Mean |Z| under attacks
ax = axes[1, 2]
for attack_name, sub in attack_df.groupby('attack'):
    style = styles.get(attack_name, 'o-')
    ax.plot(sub['strength'], sub['mean_abs_z'], style, label=attack_name, linewidth=2, markersize=7)
ax.axhline(3.0, color='black', linestyle=':', alpha=0.5, label='|Z|=3')
ax.set_xlabel('Attack Strength')
ax.set_ylabel('Mean |Z-score|')
ax.set_title('(f) Detection Signal under Attacks', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.suptitle('Assignment 3: Multi-bit Structured Residual Watermark for Time-Series/Table Foundation Models', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'assignment3_full_charts.png', dpi=180, bbox_inches='tight')
plt.show()
print("Charts saved.")


Charts saved.


## 7. 鲁棒性边界分析

确定各攻击类型下方案保持 >90% bit 恢复率的最大可承受攻击强度。

In [7]:
print("--- Robustness Boundary Analysis (bit accuracy > 0.9) ---")
for attack_name, sub in attack_df.groupby('attack'):
    valid = sub[sub['bit_accuracy'] >= 0.9]
    if len(valid) > 0:
        max_strength = valid['strength'].max()
        acc_at_max = valid[valid['strength'] == max_strength]['bit_accuracy'].values[0]
        print(f"  {attack_name}: max strength = {max_strength}, accuracy = {acc_at_max:.4f}")
    else:
        print(f"  {attack_name}: no configuration achieves >90% accuracy")


--- Robustness Boundary Analysis (bit accuracy > 0.9) ---
  additive_noise: max strength = 0.18, accuracy = 1.0000
  quantization: max strength = 3.0, accuracy = 1.0000
  row_drop: max strength = 0.6, accuracy = 1.0000
  smoothing: max strength = 1.0, accuracy = 1.0000


## 8. 方案总结与讨论

### 8.1 核心贡献

本方案将文本水印中的"密钥控制统计偏置+Z-test检测"范式成功迁移到结构化数据领域：

| 特性 | 文本水印 (KGW) | 本方案 (结构化残差水印) |
|------|---------------|----------------------|
| 嵌入域 | Token logit | 数值残差 |
| 偏置方式 | 绿名单 logit 加 $\delta$ | 残差方向加 $\delta \cdot \sigma_y$ |
| 检测方式 | Token 级 Z-test | Bit 分组 Z-test |
| 信息容量 | 1 bit (有水印/无水印) | 12 bit (4096 种身份) |
| 适用场景 | 文本生成 | 时间序列/表格预测 |

### 8.2 鲁棒性评估

- **加性噪声** ($\alpha \leq 0.08$)：高度鲁棒，|Z| 保持在 5 以上
- **随机删行** (删除率 $\leq 50\%$)：由于样本量大 (7200 行)，即使删除一半数据每个 bit 仍有足够样本
- **平滑攻击** (窗口 $\leq 7$)：对平滑较为敏感，因为平滑直接削弱残差中的偏置信号
- **量化攻击** (小数位数 $\geq 0$)：非常鲁棒，即使舍入到整数仍可完美恢复

### 8.3 部署考量

在实际表格/时间序列基础模型部署时需额外检查：
- **业务约束**：确保水印扰动不违反单调性、非负性等业务规则
- **相关矩阵保持**：多变量场景下需保持变量间相关结构
- **下游任务影响**：验证水印对预测误差、异常检测等下游任务的增量影响

### 8.4 局限性

- 仅使用合成数据验证，未在真实表格/TS基础模型输出上测试
- 水印强度需根据数据尺度和噪声水平调整
- 未考虑自适应攻击者（了解水印算法的攻击者可能设计针对性移除策略）
